# OfflineMedia Portable Video AI

This notebook is the Colab GPU worker for `CAption11/offlinemedia`. GitHub Actions creates durable generation requests; this notebook polls the queue and runs the shared OfflineMedia generation engine.

**Repository:** `CAption11/offlinemedia`  
**Branch:** `claude/scan-repo-chatgpt-review-6nljgh`  
**Trigger:** `.github/workflows/portable-trigger.yml`

In [ ]:
# 1. Clone/update the exact working branch
import os, subprocess, sys, pathlib
REPO = 'https://github.com/CAption11/offlinemedia.git'
BRANCH = 'claude/scan-repo-chatgpt-review-6nljgh'
ROOT = pathlib.Path('/content/offlinemedia')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(ROOT)
print('Repository:', ROOT)
print('Branch:', BRANCH)

In [ ]:
# 2. Colab GPU diagnostics
import subprocess, shutil
print('Python:', sys.version)
print('nvidia-smi:', shutil.which('nvidia-smi'))
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    raise RuntimeError('No NVIDIA GPU runtime detected. In Colab select Runtime > Change runtime type > GPU.')

In [ ]:
# 3. Install OfflineMedia portable dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Dependencies installed.')

## GitHub authentication

For the public repository, reading `portable/trigger.json` works without a token. A GitHub token is still recommended for private-repository use and for future job status updates.

If you use a token, put it in a Colab Secret named `GITHUB_TOKEN`. Never paste a token into a committed notebook cell.

In [ ]:
# 4. Configure optional GitHub token from Colab Secrets
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
try:
    from google.colab import userdata
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
print('GitHub token configured:', bool(GITHUB_TOKEN))

In [ ]:
# 5. Verify the trigger listener against the active branch
from portable.trigger_listener import get_trigger
trigger, trigger_sha = get_trigger(token=GITHUB_TOKEN)
print('Trigger blob SHA:', trigger_sha)
print('Trigger:', trigger)

In [ ]:
# 6. Wait for a GitHub Actions generation request
from portable.trigger_listener import wait_for_trigger
print('Waiting for a queued request...')
trigger = wait_for_trigger(token=GITHUB_TOKEN, poll_seconds=15)
print('Received trigger:', trigger)

In [ ]:
# 7. Execute the shared OfflineMedia generation engine
# The request is translated into the existing CLI so Portable and Mainload
# exercise the same backend. This cell requires a running ComfyUI server
# and a verified workflow/model installation.
mode = trigger.get('mode', 'text_to_video')
prompt = trigger.get('prompt', 'A small red ball rolling across a wooden table, natural lighting')
width = int(trigger.get('width', 320))
height = int(trigger.get('height', 240))
frames = int(trigger.get('frames', 17))
fps = int(trigger.get('fps', 8))
cmd = [sys.executable, 'scripts/test_generation.py', '--type', mode, '--prompt', prompt, '--width', str(width), '--height', str(height), '--frames', str(frames), '--fps', str(fps)]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise RuntimeError(f'Generation failed with exit code {result.returncode}')
print('Generation completed successfully.')

## Current limitation

This notebook now connects the GitHub trigger to the shared generation entry point, but a real video is only considered successful when ComfyUI, the verified Wan workflow/model, and the NVIDIA runtime all execute successfully and produce a validated output file. Configuration alone is not evidence of generation.